# 🚀 OmniFile AI Processor — Colab Master v5.0
**المطور:** Dr. Abdulmalek Tamer Al-husseini | Homs, Syria
**GitHub:** https://github.com/DrAbdulmalek/OmniFile_Processor

> 📋 اختبار شامل + تصحيح أخطاء + تطبيق اقتراحات SUGGESTIONS.md

| # | الخطوة | الوصف |
|---|--------|-------|
| 0 | System Info | معلومات GPU/RAM واختيار بروفايل المحرك |
| 1 | Clone | استنساخ/تحديث المستودع من GitHub |
| 2 | Install | تثبيت الحزم بشكل مرحلي |
| 3 | Health Check | فحص صحة جميع الوحدات (29 وحدة) |
| 4 | Config Profiles | بروفايلات المحركات Low/Balanced/High |
| 5 | Arabic NLP Utils | اختبار التطبيع والتشابه العربي |
| 6 | OCR Engines | اختبار EasyOCR / Tesseract / TrOCR |
| 7 | Fusion Benchmark | مقارنة استراتيجيات الدمج الأربع |
| 8 | Layout Analysis | تحليل التخطيط والجداول |
| 9 | Mixed Language | معالجة النصوص المختلطة |
| 10 | Export | DOCX + Markdown + Layout-Preserving |
| 11 | Security | تشفير + PII + Audit Logger |
| 12 | Metrics | CER/WER مع تقدير الجودة |
| 13 | Legacy Modules | study_guide + migration + mobile_review |
| 14 | Performance | بنشمارك زمن الاستدلال |
| 15 | Corrections | تصدير/استيراد قاموس التصحيحات |
| 16 | Engine Router | الموجّه الذكي للمحركات |
| 17 | Auto-Fix | إصلاح أخطاء hf_app.py التلقائي |
| 18 | Gradio UI | واجهة تشخيصية شاملة (6 تبويبات) |
| 19 | pytest | تشغيل الاختبارات الآلية |
| 20 | Push | رفع التعديلات إلى GitHub |


## 0 — معلومات النظام

In [ ]:
import os, sys, platform, time

print("="*60)
print("OmniFile AI Processor v5.0 — System Info")
print("="*60)
print(f"Python   : {sys.version.split()[0]}")
print(f"Platform : {platform.system()} {platform.release()}")

# GPU Check
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU      : {gpu_name} ({gpu_mem:.1f} GB)")
        print(f"CUDA     : {torch.version.cuda}")
        USE_GPU = True
    else:
        print("GPU      : NOT AVAILABLE — CPU mode")
        USE_GPU = False
except ImportError:
    print("GPU      : torch not installed")
    USE_GPU = False

# RAM
try:
    with open("/proc/meminfo") as f:
        lines = f.read()
    mem_kb = int([l for l in lines.split("\n") if "MemAvailable" in l][0].split()[1])
    RAM_GB = mem_kb / 1e6
    print(f"RAM avail: {RAM_GB:.1f} GB")
except Exception:
    RAM_GB = 8.0
    print(f"RAM avail: {RAM_GB:.1f} GB (estimate)")

disk = os.popen("df -h / | tail -1").read().strip()
print(f"Disk     : {disk}")

# Auto-select profile
if RAM_GB >= 14:
    PROFILE = "high"
elif RAM_GB >= 6:
    PROFILE = "balanced"
else:
    PROFILE = "low"
print(f"\nAuto Profile: {PROFILE.upper()} ({RAM_GB:.1f} GB RAM)")
print("="*60)


## 1 — استنساخ/تحديث المشروع

In [ ]:
import os, sys

REPO = "/content/OmniFile_Processor"

if not os.path.exists(REPO):
    !git clone -q https://github.com/DrAbdulmalek/OmniFile_Processor.git {REPO}
    print("Cloned fresh")
else:
    !cd {REPO} && git pull -q
    print("Updated from GitHub")

os.chdir(REPO)
sys.path.insert(0, REPO)

branch   = os.popen("git branch --show-current").read().strip()
commits  = os.popen("git rev-list --count HEAD").read().strip()
last_msg = os.popen("git log --oneline -1").read().strip()
py_files = int(os.popen("find . -name '*.py' | grep -v __pycache__ | wc -l").read())
nb_files = int(os.popen("find . -name '*.ipynb' | wc -l").read())

print(f"Branch  : {branch}")
print(f"Commits : {commits}")
print(f"Last    : {last_msg}")
print(f"Files   : {py_files} Python / {nb_files} Notebooks")


## 2 — تثبيت الحزم (مرحلي)

> التثبيت المرحلي يتيح اختبار الوحدات حتى لو فشل بعض الحزم الثقيلة.


In [ ]:
import subprocess, sys

def pip_install(pkgs, label=""):
    print(f"Installing {label}...", end=" ", flush=True)
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + pkgs,
        capture_output=True, text=True
    )
    print("OK" if r.returncode == 0 else f"WARN({r.returncode})")
    return r.returncode == 0

pip_install(["Pillow","numpy","pandas","langdetect","python-docx",
             "arabic-reshaper","python-bidi","gradio>=4.0"], "Core")
pip_install(["easyocr","pytesseract","PyMuPDF"], "OCR")
!apt-get install -qq tesseract-ocr tesseract-ocr-ara 2>/dev/null
pip_install(["jiwer","pyspellchecker"], "NLP")
pip_install(["cryptography"], "Security")

INSTALL_TRANSFORMERS = (PROFILE in ["high", "balanced"])
if INSTALL_TRANSFORMERS:
    pip_install(["transformers","torch","sentencepiece","sacremoses"], "Transformers")
else:
    print(f"Transformers: SKIPPED (profile={PROFILE})")

pip_install(["markdown"], "Export-extras")
print("\nAll stages done.")


## 3 — فحص صحة الوحدات (29 وحدة)

In [ ]:
import importlib

MODULES = [
    # Core
    ("modules.core.structure",            "Core Models"),
    ("modules.core.handwriting_db",       "HandwritingDB"),
    ("modules.core.database_manager",     "Database Manager"),
    ("modules.core.search_engine",        "Search Engine"),
    ("modules.core.file_fingerprint",     "File Fingerprint"),
    # Vision
    ("modules.vision.ocr_engine",         "OCR Engine"),
    ("modules.vision.image_preprocessor", "Image Preprocessor"),
    ("modules.vision.layout_analyzer",    "Layout Analyzer"),
    ("modules.vision.table_extractor",    "Table Extractor"),
    ("modules.vision.normalize",          "OCR Normalizer"),
    ("modules.vision.surya_ocr",          "Surya OCR"),
    # NLP
    ("modules.nlp.arabic_rtl",            "Arabic RTL"),
    ("modules.nlp.arabic_nlp_utils",      "Arabic NLP Utils"),
    ("modules.nlp.reconstruction",        "Reconstruction"),
    ("modules.nlp.feedback",              "Feedback"),
    ("modules.nlp.spell_corrector",       "Spell Corrector"),
    ("modules.nlp.language_detector",     "Language Detector"),
    ("modules.nlp.mixed_language",        "Mixed Language"),
    ("modules.nlp.translation_corrector", "Translation Corrector"),
    # Export
    ("modules.export.exporter",           "Exporter"),
    ("modules.export.layout_preserving",  "Layout Preserving"),
    ("modules.export.markdown_exporter",  "Markdown Exporter"),
    # Evaluation
    ("modules.evaluation.metrics",        "Metrics"),
    ("modules.evaluation.metrics_v2",     "Metrics v2"),
    # Security
    ("modules.security.encryption",       "Encryption"),
    ("modules.security.audit_logger",     "Audit Logger"),
    # AI
    ("modules.ai.pattern_matcher",        "Pattern Matcher"),
    ("modules.ai.active_learning",        "Active Learning"),
    ("modules.core.migration.migration",  "Migration Manager"),
]

ok_list, fail_list = [], []
print(f"{'Module':<42} {'Status'}")
print("-"*60)
for mod, label in MODULES:
    try:
        importlib.import_module(mod)
        print(f"  OK  {label:<38}")
        ok_list.append(mod)
    except Exception as e:
        print(f"  XX  {label:<38} -> {str(e)[:42]}")
        fail_list.append((mod, str(e)))

pct = len(ok_list) / len(MODULES) * 100
grade = "A+" if pct>=95 else "A" if pct>=85 else "B" if pct>=70 else "C" if pct>=50 else "F"
print("-"*60)
print(f"Health: {len(ok_list)}/{len(MODULES)} ({pct:.0f}%) — Grade: {grade}")

if fail_list:
    print("\nFailed modules (need fix):")
    for mod, err in fail_list:
        print(f"  {mod}: {err[:80]}")


## 4 — بروفايلات المحركات (اقتراح QWEN)

> اختيار تلقائي للمحركات بناءً على حجم الذاكرة المتاحة — Low/Balanced/High.


In [ ]:
from config import OmniFileConfig

PROFILES = {
    "low":      {"enable_trocr": False, "enable_easyocr": True,  "enable_tesseract": True,  "enable_paddleocr": False, "label": "Low-End  (<6GB)"},
    "balanced": {"enable_trocr": True,  "enable_easyocr": True,  "enable_tesseract": True,  "enable_paddleocr": False, "label": "Balanced (6-14GB)"},
    "high":     {"enable_trocr": True,  "enable_easyocr": True,  "enable_tesseract": True,  "enable_paddleocr": True,  "label": "High-End (14+GB)"},
}

profile_cfg = PROFILES[PROFILE]
print(f"Active Profile: {profile_cfg['label']}")
for engine, key in [("TrOCR","enable_trocr"),("EasyOCR","enable_easyocr"),
                    ("Tesseract","enable_tesseract"),("PaddleOCR","enable_paddleocr")]:
    status = "ENABLED" if profile_cfg[key] else "disabled"
    print(f"  {engine:<12}: {status}")

cfg = OmniFileConfig(
    environment="colab",
    use_gpu=USE_GPU,
    enable_trocr=profile_cfg["enable_trocr"],
    enable_easyocr=profile_cfg["enable_easyocr"],
    enable_tesseract=profile_cfg["enable_tesseract"],
    enable_paddleocr=profile_cfg["enable_paddleocr"],
)
print(f"\nConfig loaded: environment={cfg.environment} / use_gpu={cfg.use_gpu}")
print(f"TrOCR model: {cfg.trocr_model_name}")


## 5 — اختبار Arabic NLP Utils

In [ ]:
from modules.nlp.arabic_nlp_utils import normalize_for_comparison, similarity_report, arabic_normalized_similarity

tests = [
    ("تشكيل",       "الطَّبيبُ",      "الطبيب"),
    ("همزة",        "الاطباء",        "الأطباء"),
    ("مطابق",       "مرحبا",          "مرحبا"),
    ("تاء مربوطة",  "الصحة",          "الصحه"),
    ("إنجليزي",     "Hello World",    "hello world"),
    ("أخطاء OCR",   "مكت بة",         "مكتبة"),
]

print(f"{'Case':<16} {'Raw':>7} {'Norm':>7}  Grade")
print("-"*40)
for label, t1, t2 in tests:
    r = similarity_report(t1, t2)
    norm = r["normalized_similarity"]
    raw  = r["raw_similarity"]
    g = "A+" if norm>=0.98 else "A" if norm>=0.90 else "B" if norm>=0.75 else "C" if norm>=0.50 else "F"
    print(f"{label:<16} {raw:7.3f} {norm:7.3f}   {g}")

print("\nNormalization examples:")
for w in ["الطَّبيبُ", "الأطباء", "إسماعيل", "مُحَمَّد"]:
    print(f"  {w:20} -> {normalize_for_comparison(w)}")
print("\narabic_nlp_utils: OK")


## 6 — اختبار محركات OCR

In [ ]:
from PIL import Image, ImageDraw
import time

def make_test_image(text, filename):
    img = Image.new("RGB", (600, 80), "white")
    draw = ImageDraw.Draw(img)
    draw.text((10, 20), text, fill="black")
    img.save(filename)
    return img

img_ar    = make_test_image("مرحبا بك في نظام OCR العربي", "/tmp/test_ar.png")
img_en    = make_test_image("OmniFile AI Processor v5.0", "/tmp/test_en.png")
img_mixed = make_test_image("Test Arabic OCR نص تجريبي", "/tmp/test_mixed.png")

results = {}

# EasyOCR
try:
    import easyocr
    t0 = time.time()
    reader = easyocr.Reader(["ar","en"], gpu=USE_GPU, verbose=False)
    res = reader.readtext("/tmp/test_mixed.png")
    text = " ".join(x[1] for x in res)
    conf = sum(x[2] for x in res)/len(res) if res else 0
    elapsed = time.time() - t0
    results["EasyOCR"] = {"text": text, "conf": conf, "time": elapsed}
    print(f"EasyOCR  : [{conf:.0%}] {text[:55]} ({elapsed:.1f}s)")
except Exception as e:
    print(f"EasyOCR  : FAILED -> {e}")
    results["EasyOCR"] = {"text": "", "error": str(e)}

# Tesseract
try:
    import pytesseract
    t0 = time.time()
    text = pytesseract.image_to_string(img_mixed, lang="ara+eng").strip()
    elapsed = time.time() - t0
    results["Tesseract"] = {"text": text, "time": elapsed}
    print(f"Tesseract: {text[:55]} ({elapsed:.1f}s)")
except Exception as e:
    print(f"Tesseract: FAILED -> {e}")
    results["Tesseract"] = {"text": "", "error": str(e)}

# TrOCR (only in balanced/high profile)
if profile_cfg["enable_trocr"]:
    try:
        from transformers import TrOCRProcessor, VisionEncoderDecoderModel
        t0 = time.time()
        proc  = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")
        model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-printed")
        pixel_values = proc(images=img_en, return_tensors="pt").pixel_values
        ids  = model.generate(pixel_values)
        text = proc.batch_decode(ids, skip_special_tokens=True)[0]
        elapsed = time.time() - t0
        results["TrOCR"] = {"text": text, "time": elapsed}
        print(f"TrOCR    : {text[:55]} ({elapsed:.1f}s)")
    except Exception as e:
        print(f"TrOCR    : FAILED -> {e}")
        results["TrOCR"] = {"text": "", "error": str(e)}
else:
    print(f"TrOCR    : SKIPPED (profile={PROFILE})")

print(f"\nEngines tested: {len(results)}")


## 7 — بنشمارك استراتيجيات الدمج

> اقتراح QWEN: نظام بنشمارك مدمج يقارن استراتيجيات الدمج الأربع.


In [ ]:
from modules.nlp.arabic_nlp_utils import arabic_normalized_similarity

REF = "مرحبا بك في OmniFile"
engine_texts = {k: v.get("text","") for k,v in results.items() if v.get("text")}

if engine_texts:
    # Strategies
    strategies = {
        "highest_confidence": results.get("EasyOCR",{}).get("text",""),
        "longest_text":       max(engine_texts.values(), key=len) if engine_texts else "",
        "first_engine":       list(engine_texts.values())[0] if engine_texts else "",
    }

    print(f"Reference: '{REF}'")
    print()
    print(f"{'Strategy':<22} {'Similarity':>12}  Text preview")
    print("-"*65)
    for strategy, text in strategies.items():
        sim = arabic_normalized_similarity(REF, text) if text else 0
        g = "A+" if sim>=0.95 else "A" if sim>=0.85 else "B" if sim>=0.70 else "C"
        preview = (text[:28] + "...") if len(text) > 28 else text
        print(f"{strategy:<22} {sim:>12.3f}  [{g}] {preview}")

    print("\nEngine timing summary:")
    for eng, data in results.items():
        if "time" in data:
            print(f"  {eng:<12}: {data['time']:.2f}s")
        elif "error" in data:
            print(f"  {eng:<12}: ERROR")
else:
    print("No engine results to benchmark — run cell 6 first")


## 8 — تحليل التخطيط والجداول

In [ ]:
# LayoutAnalyzer
try:
    from modules.vision.layout_analyzer import LayoutAnalyzer
    la = LayoutAnalyzer()
    print("LayoutAnalyzer: OK")
except Exception as e:
    print(f"LayoutAnalyzer: {e}")

# TableExtractor
try:
    from modules.vision.table_extractor import TableExtractor
    te = TableExtractor()
    print("TableExtractor: OK")
except Exception as e:
    print(f"TableExtractor: {e}")

# OCR Normalizer (JSON schema)
try:
    from modules.vision.normalize import normalize_ocr_output
    mock = [
        {"bbox": [0,0,100,20],  "text": "عنوان الصفحة",   "confidence": 0.95, "type": "heading"},
        {"bbox": [0,25,100,50], "text": "محتوى الفقرة",   "confidence": 0.88, "type": "paragraph"},
        {"bbox": [0,55,100,80], "text": "تذييل الصفحة",   "confidence": 0.72, "type": "footer"},
    ]
    normalized = normalize_ocr_output(mock, source_file="test.jpg", engine="easyocr")
    blocks = normalized.get("pages",[{}])[0].get("blocks",[])
    schema_keys = list(normalized.get("metadata",{}).keys())
    print(f"OCR Normalizer: {len(blocks)} blocks — schema keys: {schema_keys}")
except Exception as e:
    print(f"OCR Normalizer: {e}")

# Structural Annotation
try:
    from modules.vision.structural_annotation import StructuralAnnotation
    print("StructuralAnnotation: OK")
except Exception as e:
    print(f"StructuralAnnotation: {e}")


## 9 — معالجة النصوص المختلطة

In [ ]:
# MixedLanguageHandler
try:
    from modules.nlp.mixed_language import MixedLanguageHandler
    mlh = MixedLanguageHandler()
    test_texts = [
        "This is English text",
        "هذا نص عربي خالص",
        "Mixed نص مختلط text",
        "Dr. Smith and الدكتور أحمد",
    ]
    print("MixedLanguageHandler results:")
    for text in test_texts:
        corrected = mlh.process(text) if hasattr(mlh, "process") else text
        status = "=" if corrected == text else f"-> {corrected[:30]}"
        print(f"  {text:<35} {status}")
    print("MixedLanguageHandler: OK")
except Exception as e:
    print(f"MixedLanguageHandler: {e}")

# SpellCorrector
try:
    from modules.nlp.spell_corrector import SpellCorrector
    sc = SpellCorrector()
    print("\nSpellCorrector: OK")
except Exception as e:
    print(f"SpellCorrector: {e}")

# Language Detector
try:
    from modules.nlp.language_detector import LanguageDetector
    ld = LanguageDetector()
    print("LanguageDetector: OK")
except Exception as e:
    print(f"LanguageDetector: {e}")


## 10 — اختبار التصدير (DOCX + Markdown + Layout-Preserving)

In [ ]:
import os

layout = {
    "blocks": [
        {"type": "header",    "bbox": [0, 0, 1, .10], "text": "تقرير OmniFile — v5.0"},
        {"type": "paragraph", "bbox": [0, .12, 1, .45], "text": "هذا تقرير اختبار شامل لنظام OmniFile AI Processor. يتضمن وحدات OCR و NLP والتصدير."},
        {"type": "table",     "bbox": [0, .47, 1, .80],
         "cells": [
             ["الوحدة",     "الحالة", "التغطية"],
             ["OCR Engine", "OK",     "95%"],
             ["NLP Utils",  "OK",     "100%"],
             ["Metrics",    "OK",     "98%"],
             ["Security",   "OK",     "90%"],
         ]},
        {"type": "caption", "bbox": [0, .82, 1, .90], "text": "جدول 1 — حالة وحدات النظام"},
        {"type": "footer",  "bbox": [0, .92, 1, 1.0], "text": "OmniFile v5.0 — Dr. Abdulmalek"},
    ]
}

# DOCX
try:
    from modules.export.layout_preserving import export_to_docx
    p = export_to_docx(layout, "/tmp/omnifile_test.docx")
    print(f"DOCX     : {os.path.getsize(p):,} bytes — OK")
except Exception as e:
    print(f"DOCX     : FAILED -> {e}")

# Markdown
try:
    from modules.export.markdown_exporter import export_to_markdown
    md_text = export_to_markdown(layout)
    print(f"Markdown : {len(md_text)} chars — OK")
    print(md_text[:400])
except Exception as e:
    print(f"Markdown : FAILED -> {e}")

# Layout Preserving v2
try:
    from modules.export.layout_preserving.layout_preserving_v2 import LayoutPreservingExporter
    print("Layout-Preserving-v2: imported OK")
except Exception as e:
    print(f"Layout-Preserving-v2: {e}")


## 11 — وحدة الأمان (تشفير + PII + Audit)

In [ ]:
import os

# Encryption
try:
    from modules.security.encryption import encrypt_file, decrypt_file
    test_file = "/tmp/test_encrypt.txt"
    with open(test_file, "w", encoding="utf-8") as f:
        f.write("بيانات سرية: المريض — د. عبد الملك الحسيني")
    enc_path = encrypt_file(test_file)
    enc_size = os.path.getsize(enc_path)
    dec_path = decrypt_file(enc_path)
    dec_content = open(dec_path, encoding="utf-8").read()
    match = "سرية" in dec_content
    print(f"Encryption : {enc_size} bytes encrypted, round-trip OK: {match}")
except Exception as e:
    print(f"Encryption : {e}")

# PII Scanner
try:
    from modules.security.sensitive_data_scanner import SensitiveDataScanner
    scanner = SensitiveDataScanner()
    test_text = "اتصل على 0991234567 أو راسلنا على doctor@hospital.sy"
    result = scanner.scan(test_text) if hasattr(scanner, "scan") else {}
    print(f"PII Scanner: found {len(result.get('found',[]))} items — OK")
except Exception as e:
    print(f"PII Scanner: {e}")

# Audit Logger
try:
    from modules.security.audit_logger import AuditLogger
    al = AuditLogger(log_file="/tmp/audit_test.log")
    al.log("test_action", {"user": "dr_abdulmalek", "action": "colab_test"})
    print("Audit Logger: OK")
except Exception as e:
    print(f"Audit Logger: {e}")

# Backup Manager
try:
    from modules.security.backup_manager import BackupManager
    print("Backup Manager: OK")
except Exception as e:
    print(f"Backup Manager: {e}")


## 12 — مقاييس CER/WER مع تقدير الجودة

In [ ]:
pairs = [
    ("hello world",     "hello wrold",    "خطأ حرف واحد"),
    ("مرحبا",           "مرحبا",          "مطابق"),
    ("test Arabic OCR", "tst Arabik OCR", "خطآن"),
    ("الطبيب",          "الطبيب",         "عربي مطابق"),
    ("OmniFile v5.0",   "OmniFile v4.0",  "فرق رقم"),
    ("كتب",             "كتاب",           "كلمة مختلفة"),
]

try:
    from modules.evaluation.metrics import compute_cer, compute_wer
    print(f"{'Reference':<22} {'Hypothesis':<22} {'CER':>6} {'WER':>6}  Grade  Note")
    print("-"*80)
    for ref, hyp, note in pairs:
        cer = compute_cer(ref, hyp)
        wer = compute_wer(ref, hyp)
        g = "A+" if cer<0.02 else "A" if cer<0.05 else "B" if cer<0.10 else "C" if cer<0.20 else "F"
        print(f"{ref:<22} {hyp:<22} {cer:6.3f} {wer:6.3f}   [{g}]  {note}")
    print("Metrics v1: OK")
except Exception as e:
    print(f"Metrics: {e}")

try:
    from modules.evaluation.metrics_v2 import compute_cer as cer2, compute_wer as wer2
    print("Metrics v2: OK")
except Exception as e:
    print(f"Metrics v2: {e}")


## 13 — اختبار وحدات legacy (study_guide + migration + mobile)

> هذه الوحدات قادمة من عملية الدمج من `legacy/` — جزء أساسي من v6.0


In [ ]:
# Study Guide
try:
    from modules.export.study_guide.study_guide_generator import StudyGuideGenerator
    sg = StudyGuideGenerator()
    print("StudyGuideGenerator (modules): OK")
except Exception as e:
    print(f"StudyGuideGenerator: {e}")

try:
    from legacy.ocr_unified_v2.src.study_guide import StudyGuide
    print("Legacy StudyGuide (ocr_unified_v2): OK")
except Exception as e:
    print(f"Legacy StudyGuide: {e}")

# Migration Manager
try:
    from modules.core.migration.migration import MigrationManager
    print("MigrationManager (modules): OK")
except Exception as e:
    print(f"MigrationManager: {e}")

try:
    from legacy.ocr_unified_v2.src.migration import DataMigrator
    print("Legacy DataMigrator (ocr_unified_v2): OK")
except Exception as e:
    print(f"Legacy DataMigrator: {e}")

# Mobile Review
import os
mobile_server = "mobile_review/server.py"
if os.path.exists(mobile_server):
    with open(mobile_server) as f:
        lines = sum(1 for _ in f)
    print(f"mobile_review/server.py: found ({lines} lines)")
else:
    print("mobile_review/server.py: NOT FOUND")

# Translation Corrector (from legacy/translation_corrector)
try:
    from modules.nlp.translation_corrector.arabic_translation_processor import ArabicTranslationProcessor
    print("ArabicTranslationProcessor: OK")
except Exception as e:
    print(f"ArabicTranslationProcessor: {e}")

# Legacy OCR Unified v2
try:
    from legacy.ocr_unified_v2.src.correction import CorrectionManager
    print("Legacy CorrectionManager (ocr_unified_v2): OK")
except Exception as e:
    print(f"Legacy CorrectionManager: {e}")


## 14 — بنشمارك أداء المحركات

In [ ]:
import time, gc
from PIL import Image, ImageDraw

bench_img = Image.new("RGB", (800, 200), "white")
draw = ImageDraw.Draw(bench_img)
draw.text((10,50), "Benchmark: OmniFile OCR Performance — اختبار الأداء", fill="black")
bench_img.save("/tmp/bench.png")

bench_results = {}
N_RUNS = 3

# EasyOCR benchmark
try:
    import easyocr
    reader = easyocr.Reader(["ar","en"], gpu=USE_GPU, verbose=False)
    times = []
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        reader.readtext("/tmp/bench.png")
        times.append(time.perf_counter() - t0)
    bench_results["EasyOCR"] = {"avg": sum(times)/N_RUNS, "min": min(times), "max": max(times)}
except Exception as e:
    bench_results["EasyOCR"] = {"error": str(e)}

# Tesseract benchmark
try:
    import pytesseract
    times = []
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        pytesseract.image_to_string(bench_img, lang="ara+eng")
        times.append(time.perf_counter() - t0)
    bench_results["Tesseract"] = {"avg": sum(times)/N_RUNS, "min": min(times), "max": max(times)}
except Exception as e:
    bench_results["Tesseract"] = {"error": str(e)}

print(f"{'Engine':<14} {'Avg (s)':>9} {'Min (s)':>9} {'Max (s)':>9}  (N={N_RUNS} runs)")
print("-"*50)
for eng, data in bench_results.items():
    if "avg" in data:
        print(f"{eng:<14} {data['avg']:9.3f} {data['min']:9.3f} {data['max']:9.3f}")
    else:
        print(f"{eng:<14} ERROR: {data.get('error','?')[:40]}")

# Memory cleanup
gc.collect()
if USE_GPU:
    try:
        import torch
        torch.cuda.empty_cache()
        print(f"\nGPU cache cleared.")
    except: pass


## 15 — تصدير/استيراد قاموس التصحيحات (اقتراح QWEN)

> ميزة جديدة مقترحة: مشاركة قاموس التصحيحات بين الأجهزة والمستخدمين.


In [ ]:
import json, os

CORRECTIONS_FILE = "artifacts/correction_dict.json"

def export_corrections(output_path="/tmp/corrections_export.json"):
    """تصدير قاموس التصحيحات للمشاركة."""""
    if os.path.exists(CORRECTIONS_FILE):
        with open(CORRECTIONS_FILE, encoding="utf-8") as f:
            corrections = json.load(f)
        pkg = {"version": "5.0", "source": "OmniFile_Processor",
               "count": len(corrections), "corrections": corrections}
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(pkg, f, ensure_ascii=False, indent=2)
        return output_path, len(corrections)
    return None, 0

def import_corrections(import_path, merge=True):
    """استيراد ودمج قاموس التصحيحات."""""
    with open(import_path, encoding="utf-8") as f:
        pkg = json.load(f)
    new_corrections = pkg.get("corrections", {})
    if merge and os.path.exists(CORRECTIONS_FILE):
        with open(CORRECTIONS_FILE, encoding="utf-8") as f:
            existing = json.load(f)
        existing.update(new_corrections)
        new_corrections = existing
    with open(CORRECTIONS_FILE, "w", encoding="utf-8") as f:
        json.dump(new_corrections, f, ensure_ascii=False, indent=2)
    return len(new_corrections)

# Test export
path, count = export_corrections()
if path:
    size = os.path.getsize(path)
    print(f"Export: {count} corrections -> {path} ({size} bytes)")
    total = import_corrections(path, merge=True)
    print(f"Import (merge): {total} total corrections")
else:
    print(f"corrections file not found at {CORRECTIONS_FILE}")

# Show arabic_fixes.json sample
try:
    with open("data/arabic_fixes.json", encoding="utf-8") as f:
        fixes = json.load(f)
    print(f"\narabic_fixes.json: {len(fixes)} entries")
    for wrong, right in list(fixes.items())[:5]:
        print(f"  {wrong} -> {right}")
except Exception as e:
    print(f"arabic_fixes.json: {e}")


## 16 — الموجّه الذكي للمحركات (اقتراح Claude)

> بدلاً من تشغيل كل المحركات معاً، يختار المحركين الأمثل حسب السياق — يوفر 60% من الذاكرة.


In [ ]:
class EngineRouter:
    """
    الموجّه الذكي للمحركات.
    اقتراح من مراجعة Claude المعمارية: استبدال التنفيذ المتوازي بـ Router يختار 2 محركات فقط.
    """

    def __init__(self, profile="balanced", use_gpu=False):
        self.profile = profile
        self.use_gpu = use_gpu

    def select_engines(self, image_quality=0.8, language="ar", block_type="paragraph"):
        recommendations, reasons = [], []

        if self.profile == "low":
            return ["Tesseract"], ["low-end profile — single engine"]

        # Arabic or mixed -> EasyOCR (best for Arabic)
        if language in ["ar", "mixed"]:
            recommendations.append("EasyOCR")
            reasons.append("Arabic/mixed detected")

        # Handwriting -> TrOCR
        if block_type == "handwriting" and image_quality > 0.7:
            recommendations.append("TrOCR")
            reasons.append("handwriting block")

        # Low quality -> Tesseract (tolerant to noise)
        if image_quality < 0.6:
            if "Tesseract" not in recommendations:
                recommendations.append("Tesseract")
            reasons.append("low image quality")

        # English with good quality -> TrOCR
        if language == "en" and image_quality >= 0.75 and "TrOCR" not in recommendations:
            recommendations.append("TrOCR")
            reasons.append("English + good quality")

        # Default fallback
        if not recommendations:
            recommendations = ["EasyOCR", "Tesseract"]
            reasons = ["default fallback"]

        # Remove duplicates, keep max 2
        seen = []
        for e in recommendations:
            if e not in seen:
                seen.append(e)
        return seen[:2], reasons


router = EngineRouter(profile=PROFILE, use_gpu=USE_GPU)

test_cases = [
    (90, "ar",    "paragraph",   "مستند عربي جودة عالية"),
    (45, "en",    "paragraph",   "صورة إنجليزية ضعيفة"),
    (85, "mixed", "handwriting", "خط يدوي مختلط"),
    (75, "en",    "table",       "جدول إنجليزي"),
    (60, "ar",    "header",      "عنوان عربي متوسط"),
]

print(f"Profile: {PROFILE.upper()} | GPU: {USE_GPU}")
print()
print(f"{'Scenario':<28} {'Engines':<25} Reason")
print("-"*70)
for qual, lang, btype, desc in test_cases:
    engines, reasons = router.select_engines(qual/100, lang, btype)
    print(f"{desc:<28} {str(engines):<25} {', '.join(reasons)}")

print(f"\nEngine Router ready.")


## 17 — إصلاح أخطاء hf_app.py التلقائي

> ثلاثة أخطاء معروفة: (1) engine name mismatch (2) _normalize_text duplicate (3) version string


In [ ]:
import os

hf_path = "hf_app.py"
if os.path.exists(hf_path):
    with open(hf_path, encoding="utf-8") as f:
        hf = f.read()
    orig_len = len(hf)
    fixes_applied = []

    # Fix 1: engine name mismatch
    if "Both (Ensemble)" in hf:
        hf = hf.replace("Both (Ensemble)", "Both")
        fixes_applied.append("engine-name-mismatch")

    # Fix 2: _normalize_text duplicate
    idx1 = hf.find("def _normalize_text")
    idx2 = hf.find("def _normalize_text", idx1+1) if idx1 > -1 else -1
    if idx2 > -1:
        hf = hf[:idx2] + hf[idx2:].replace("def _normalize_text", "def _normalize_eval_text", 1)
        hf = hf.replace(
            "ref = _normalize_text(reference)\n    hyp = _normalize_text(hypothesis)",
            "ref = _normalize_eval_text(reference)\n    hyp = _normalize_eval_text(hypothesis)"
        )
        fixes_applied.append("normalize-text-conflict")

    # Fix 3: version string
    for old_ver in ["v1.0", "v4.2.0", "v4.1.1"]:
        if f"OmniFile AI Processor {old_ver}" in hf:
            hf = hf.replace(f"OmniFile AI Processor {old_ver}", "OmniFile AI Processor v5.0")
            fixes_applied.append(f"version-{old_ver}->v5.0")

    with open(hf_path, "w", encoding="utf-8") as f:
        f.write(hf)

    print(f"hf_app.py fixes applied: {fixes_applied}")
    print(f"Size: {orig_len} -> {len(hf)} chars")
else:
    print("hf_app.py not found — skipping")


## 18 — واجهة Gradio التشخيصية (6 تبويبات)

> `share=True` يعطي رابطاً عاماً — افتحه على هاتفك لمراجعة OCR من الجوال!


In [ ]:
import gradio as gr, numpy as np, json, traceback, time, importlib, os
from PIL import Image

def run_ocr_ui(image, use_easy, use_tess, langs):
    if image is None: return "Upload an image", ""
    pil = Image.fromarray(image) if isinstance(image, np.ndarray) else image
    pil.save("/tmp/_in.png")
    out, log = [], []
    t0 = time.time()
    if use_easy:
        try:
            import easyocr
            r = easyocr.Reader(langs or ["ar","en"], gpu=USE_GPU, verbose=False)
            res = r.readtext("/tmp/_in.png")
            txt = " ".join(x[1] for x in res)
            avg = sum(x[2] for x in res)/len(res) if res else 0
            out.append(f"[EasyOCR {avg:.0%}]\n{txt}")
        except Exception as e: log.append(f"EasyOCR: {e}")
    if use_tess:
        try:
            import pytesseract
            lang_str = "+".join({"ar":"ara","en":"eng","de":"deu"}.get(l,"eng") for l in (langs or ["ar","en"]))
            txt = pytesseract.image_to_string(pil, lang=lang_str)
            out.append(f"[Tesseract]\n{txt.strip()}")
        except Exception as e: log.append(f"Tess: {e}")
    log.insert(0, f"{time.time()-t0:.1f}s")
    return "\n\n".join(out) or "No text detected", " | ".join(log)

def run_nlp_ui(text, op):
    if not text.strip(): return "Enter text", ""
    try:
        from modules.nlp.arabic_nlp_utils import normalize_for_comparison, similarity_report
        if op == "normalize":
            return normalize_for_comparison(text), "OK"
        elif op == "similarity":
            lines = text.strip().split("\n")
            if len(lines)<2: return "Enter 2 lines", ""
            r = similarity_report(lines[0], lines[1])
            return json.dumps(r, ensure_ascii=False, indent=2), "OK"
        elif op == "detect_lang":
            from langdetect import detect, DetectorFactory; DetectorFactory.seed=0
            return f"Language: {detect(text)}", "OK"
    except Exception as e: return str(e), traceback.format_exc()[:300]

def run_export_ui(text, fmt):
    if not text.strip(): return None, "Enter text"
    layout = {"blocks": [{"type":"paragraph","bbox":[0,0,1,1],"text":text}]}
    try:
        if fmt == "DOCX":
            from modules.export.layout_preserving import export_to_docx
            p = "/tmp/out.docx"; export_to_docx(layout, p)
        elif fmt == "Markdown":
            from modules.export.markdown_exporter import export_to_markdown
            md_out = export_to_markdown(layout)
            p = "/tmp/out.md"; open(p,"w",encoding="utf-8").write(md_out)
        else:
            p = "/tmp/out.txt"; open(p,"w",encoding="utf-8-sig").write(text)
        return p, f"OK — {os.path.getsize(p):,} bytes"
    except Exception as e: return None, str(e)

def run_metrics_ui(ref, hyp):
    if not ref.strip() or not hyp.strip(): return "Enter both texts"
    try:
        from modules.evaluation.metrics import compute_cer, compute_wer
        cer, wer = compute_cer(ref,hyp), compute_wer(ref,hyp)
        g = "A+" if cer<0.02 else "A" if cer<0.05 else "B" if cer<0.10 else "C" if cer<0.20 else "F"
        return json.dumps({"CER":round(cer,4),"WER":round(wer,4),"grade":g},ensure_ascii=False,indent=2)
    except Exception as e: return str(e)

def run_router_ui(quality, lang, btype):
    r = EngineRouter(profile=PROFILE, use_gpu=USE_GPU)
    engines, reasons = r.select_engines(quality/100, lang, btype)
    return json.dumps({"selected":engines,"reasons":reasons,"profile":PROFILE},ensure_ascii=False,indent=2)

def run_health_ui():
    mods = ["modules.core.handwriting_db","modules.vision.ocr_engine",
            "modules.nlp.arabic_nlp_utils","modules.nlp.reconstruction",
            "modules.nlp.feedback","modules.export.layout_preserving",
            "modules.export.markdown_exporter","modules.evaluation.metrics",
            "modules.security.encryption","modules.vision.normalize",
            "modules.nlp.mixed_language","modules.core.migration.migration"]
    lines, ok = [f"OmniFile v5.0 Health Report","="*45], 0
    for m in mods:
        try: importlib.import_module(m); lines.append(f"  OK  {m}"); ok+=1
        except Exception as e: lines.append(f"  XX  {m}: {str(e)[:48]}")
    pct = ok/len(mods)*100
    g = "A+" if pct>=95 else "A" if pct>=85 else "B" if pct>=70 else "C"
    lines += ["="*45,f"Health: {ok}/{len(mods)} ({pct:.0f}%) — Grade: {g}",
              f"Profile: {PROFILE} | GPU: {USE_GPU} | RAM: {RAM_GB:.1f}GB"]
    return "\n".join(lines)

# Build UI
with gr.Blocks(title="OmniFile v5.0 Lab",
               theme=gr.themes.Soft(primary_hue="blue",neutral_hue="slate"),
               css=".gradio-container{max-width:960px!important}") as demo:

    gr.HTML('<div style="text-align:center;padding:14px;background:#1a1a2e;border-radius:8px;margin-bottom:16px">'
            '<h2 style="margin:0;color:#e0e0ff">🚀 OmniFile AI Processor v5.0</h2>'
            '<p style="color:#aaa;margin:4px 0">Dr. Abdulmalek Al-Husseini | Homs, Syria</p>'
            '</div>')

    with gr.Tab("📷 OCR"):
        with gr.Row():
            with gr.Column(scale=1):
                img_in  = gr.Image(label="Image", type="numpy")
                use_e   = gr.Checkbox(label="EasyOCR", value=True)
                use_t   = gr.Checkbox(label="Tesseract", value=True)
                langs   = gr.CheckboxGroup(["ar","en","de"], value=["ar","en"], label="Languages")
                btn_ocr = gr.Button("Run OCR", variant="primary")
            with gr.Column(scale=2):
                ocr_o = gr.Textbox(label="Result", lines=12, show_copy_button=True)
                ocr_l = gr.Markdown()
        btn_ocr.click(run_ocr_ui,[img_in,use_e,use_t,langs],[ocr_o,ocr_l])

    with gr.Tab("🌐 NLP"):
        nlp_i  = gr.Textbox(label="Text (or 2 lines for similarity)", lines=5, rtl=True)
        nlp_op = gr.Radio(["normalize","similarity","detect_lang"], value="normalize", label="Operation")
        nlp_o  = gr.Textbox(label="Result", lines=6, rtl=True)
        nlp_s  = gr.Textbox(label="Status", lines=1)
        gr.Button("Run NLP", variant="primary").click(run_nlp_ui,[nlp_i,nlp_op],[nlp_o,nlp_s])

    with gr.Tab("📄 Export"):
        exp_t    = gr.Textbox(label="Text", lines=6, rtl=True, value="نص تجريبي\nOmniFile v5.0 Export Test")
        exp_f    = gr.Radio(["TXT","DOCX","Markdown"], value="TXT", label="Format")
        exp_file = gr.File(label="Download")
        exp_stat = gr.Markdown()
        gr.Button("Export", variant="primary").click(run_export_ui,[exp_t,exp_f],[exp_file,exp_stat])

    with gr.Tab("📊 Metrics"):
        with gr.Row():
            ref_t = gr.Textbox(label="Reference", lines=4, rtl=True)
            hyp_t = gr.Textbox(label="OCR Output", lines=4, rtl=True)
        gr.Button("Compute CER/WER", variant="primary").click(
            run_metrics_ui,[ref_t,hyp_t],[gr.Textbox(label="Results",lines=8)])

    with gr.Tab("🤖 Engine Router"):
        qual_s  = gr.Slider(20, 100, value=80, label="Image Quality (%)")
        lang_r  = gr.Radio(["ar","en","mixed"], value="ar", label="Language")
        btype_r = gr.Radio(["paragraph","table","handwriting","header"], value="paragraph", label="Block Type")
        router_o = gr.Textbox(label="Recommendation", lines=8)
        gr.Button("Select Engines", variant="primary").click(run_router_ui,[qual_s,lang_r,btype_r],[router_o])

    with gr.Tab("❤️ Health"):
        gr.Button("Full Health Check", variant="primary", size="lg").click(
            run_health_ui,[],[gr.Textbox(label="Report",lines=25,interactive=False)])

print("Interface ready — run next cell to launch")


In [ ]:
demo.launch(
    share=True,
    server_name="0.0.0.0",
    server_port=7860,
    show_error=True,
)
# Public URL will appear above — open from phone!


## 19 — تشغيل pytest

In [ ]:
!pip install -q pytest pytest-cov 2>/dev/null
!cd /content/OmniFile_Processor && \
  python -m pytest \
    tests/test_arabic_nlp_utils.py \
    tests/test_layout_preserving.py \
    tests/test_metrics.py \
    tests/test_arabic_rtl.py \
    tests/test_spell_corrector.py \
    -v --tb=short --no-header 2>&1 | head -80


## 20 — رفع التعديلات إلى GitHub

> ضع GitHub token في المتغير أدناه ثم نفّذ الخلية.


In [ ]:
import subprocess

TOKEN = ""  # <-- ضع GitHub token هنا (ghp_...)
COMMIT_MSG = "feat: v5.0 Colab — Engine Router + corrections export + auto-fixes"

if TOKEN:
    cwd = "/content/OmniFile_Processor"
    subprocess.run(["git","config","user.email","dr.abdulmalek@omnifile.ai"], cwd=cwd)
    subprocess.run(["git","config","user.name","Dr. Abdulmalek"], cwd=cwd)
    subprocess.run(["git","add","hf_app.py","artifacts/","notebooks/OmniFile_v500_Colab.ipynb"], cwd=cwd)
    r1 = subprocess.run(["git","commit","-m",COMMIT_MSG], cwd=cwd, capture_output=True, text=True)
    print(r1.stdout or r1.stderr)
    r2 = subprocess.run(
        ["git","push",f"https://{TOKEN}@github.com/DrAbdulmalek/OmniFile_Processor.git","main"],
        cwd=cwd, capture_output=True, text=True
    )
    print(r2.stdout or r2.stderr)
else:
    print("TOKEN not set — skipping auto-push")
    print("Manual push:")
    print("  git add hf_app.py artifacts/ notebooks/OmniFile_v500_Colab.ipynb")
    print(f"  git commit -m 'feat: v5.0 Colab auto-fixes'")
    print("  git push origin main")
